# Email Agent

- Authenticates User: 
    - Only then are they allowed  into the "inbox"
    - Dynamic Tools and Prompt on the condition of there being an email and password in state that match hardcoded
- Checks "inbox"
    - Email In Tool
- Sends emails
    - Human in the loop

In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from dataclasses import dataclass

@dataclass
class EmailContext:
    email_address: str = "jude@example.com"
    password: str = "password123"

In [3]:
from langchain.agents import AgentState

class AuthenticatedState(AgentState):
    authenticated: bool

In [13]:
from langchain.tools import tool, ToolRuntime
from langgraph.types import Command
from langchain.messages import ToolMessage

@tool
def check_inbox() -> str:
    """Check the inbox for recent emails"""
    return """
    Hi Jude,
    I am going to be in town next week and was wondering if we couldgrab a coffee?
    - Best, Jane (jane@example.com)
    """

@tool
def send_email(to: str, subject: str, body: str) -> str:
    """Send a response email"""
    return f"Email sent to {to} with subject {subject} and body {body}"

@tool
def authenticate(email: str, password: str, runtime: ToolRuntime) -> Command:
    """Authenticate the user with the given email and password"""
    if email == runtime.context.email_address and password == runtime.context.password:
        return Command(update={
            "authenticated": True,
            "messages": [ToolMessage(
                "Successfully authenticated",
                tool_call_id=runtime.tool_call_id
            )]
        })
    else:
        return Command(
            update={
                "authenticated":False,
                "messages":[ToolMessage("Authentication Failed",tool_call_id=runtime.tool_call_id)]
            }
        )


In [5]:
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from typing import Callable

@wrap_model_call
def dynamic_tool_call(request: ModelRequest,
handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:
    """Allow read inbox and send email tools only if user providers correct email and password"""

    authenticated = request.state.get("authenticated")
    if authenticated:
        tools = [check_inbox, send_email]
    else:
        tools = [authenticate]

    request = request.override(tools=tools)
    return handler(request)


In [6]:
from langchain.agents.middleware import dynamic_prompt

authenticated_prompt = "You are a helpful assistant that can check the inbox and send emails."
unauthenticated_prompt = "You are a helpful assistantt that can authenticate users."

@dynamic_prompt
def dynamic_prompt(request: ModelRequest) -> str:
    """Generate system prompt based on authentitcation status"""
    authenticated = request.state.get("authenticated")

    if authenticated:
        return authenticated_prompt
    else:
        return unauthenticated_prompt

In [8]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import HumanInTheLoopMiddleware

agent = create_agent(
    model="gpt-5-nano",
    tools=[authenticate, check_inbox, send_email],
    checkpointer=InMemorySaver(),
    state_schema=AuthenticatedState,
    context_schema=EmailContext,
    middleware=[
        dynamic_tool_call,
        dynamic_prompt,
        HumanInTheLoopMiddleware(
            interrupt_on={
                "authenticate": False,
                "check_inbox":False,
                "send_email":True
            })
    ]
)

In [14]:
from langchain.messages import HumanMessage

config = {"configurable":{"thread_id":"1"}}

response = agent.invoke(
    {"messages":[HumanMessage(content="jude@example.com, password123")]},
    context=EmailContext(),
    config=config
)

print(response['messages'][-1].content)

e:\code\langchain-basics\.venv\Lib\site-packages\pydantic\main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=EmailContext(email_addres... password='password123'), input_type=EmailContext])
  return self.__pydantic_serializer__.to_python(


I can help with the email, but I can’t store or reuse passwords or log in to your account. If you’d like me to send the kickoff email to Jude, I’ll need a few details or your approval to use a ready-to-send version with placeholders filled in.

What I can do next:
- Option A (recommended): I prepare a concrete, ready-to-send kickoff email using your project details. Please provide:
  - Project Name
  - Kickoff date/time and timezone
  - Meeting link or location
  - Milestones (names/dates)
  - Your name, title, and contact info
  - Any link to the project charter or brief
  - RSVP method and kickoff deadline
  - Any specific notes you want included
  I’ll then present the full email for your review and, on your OK, send it to Jude at Jude’s address.

- Option B: I generate a filled-in sample draft (using placeholder-friendly example values) you can customize before sending. This is useful if you want a quick, editable version right away.

- Option C: I can just draft the email content 

In [15]:
# Send Option B to the user

config = {"configurable":{"thread_id":"1"}}

response = agent.invoke(
    {"messages":[HumanMessage(content="Option B - send with whatever values you want, just fill and send")]},
    context=EmailContext(),
    config=config
)

print(response['messages'][-1].content)

In [16]:
print(response)

{'messages': [HumanMessage(content='draft 1', additional_kwargs={}, response_metadata={}, id='22a7cafe-a99d-41f6-8604-c807b0fa7ead'), AIMessage(content='Could you clarify what you want drafted as “Draft 1”? Here are some quick options I can provide, plus what I’d need from you:\n\nPossible draft types\n- Email (welcome, follow-up, announcement)\n- Meeting agenda or minutes\n- Project proposal or plan\n- Policy or procedure document\n- Contract or NDA (short form)\n- Script or presentation notes\n\nInformation I need\n- Type of draft\n- Purpose and audience\n- Key points or sections to include\n- Desired length (word count or pages)\n- Tone (formal, neutral, friendly)\n- Any deadline or delivery date\n\nIf you’d like, I can also provide a ready-to-use generic “Draft 1” for a common item (e.g., a welcome email or a project kickoff email) and you can customize from there. Which would you prefer?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens':

In [19]:
# Print interrupt value

print(response['__interrupt__'][0].value)

{'action_requests': [{'name': 'send_email', 'args': {'to': 'jude@example.com', 'subject': 'Kickoff for Atlas Marketing Initiative – Welcome Aboard', 'body': 'Hi Jude,\n\nWelcome to the Atlas Marketing Initiative project. We’re excited to have you on the team. This kickoff email outlines our goals, timeline, roles, and next steps.\n\nProject objective\n- To launch a coordinated marketing campaign for Q2 with improved lead quality and increased conversions.\n\nKey milestones\n- Milestone 1: Charter approved by Feb 20, 2026\n- Milestone 2: Content assets delivered by Feb 28, 2026\n- Milestone 3: Campaign go-live by Mar 15, 2026\n\nYour role\n- Marketing Analyst: Data collection, performance tracking, and reporting\n- Reporting to: Maria Chen, Marketing Lead\n\nTeam\n- Maria Chen – Marketing Lead\n- Carlos Diaz – Content Manager\n- Priya Singh – Graphic Designer\n\nKickoff meeting\n- Date: Feb 12, 2026\n- Time: 10:00 AM America/New_York\n- Link/Location: https://example.com/kickoff-meeting

In [22]:
print(f"Subject: {response['__interrupt__'][0].value['action_requests'][0]['args']['subject']}")
print(f"\nBody: {response['__interrupt__'][0].value['action_requests'][0]['args']['body']}")

Subject: Kickoff for Atlas Marketing Initiative – Welcome Aboard

Body: Hi Jude,

Welcome to the Atlas Marketing Initiative project. We’re excited to have you on the team. This kickoff email outlines our goals, timeline, roles, and next steps.

Project objective
- To launch a coordinated marketing campaign for Q2 with improved lead quality and increased conversions.

Key milestones
- Milestone 1: Charter approved by Feb 20, 2026
- Milestone 2: Content assets delivered by Feb 28, 2026
- Milestone 3: Campaign go-live by Mar 15, 2026

Your role
- Marketing Analyst: Data collection, performance tracking, and reporting
- Reporting to: Maria Chen, Marketing Lead

Team
- Maria Chen – Marketing Lead
- Carlos Diaz – Content Manager
- Priya Singh – Graphic Designer

Kickoff meeting
- Date: Feb 12, 2026
- Time: 10:00 AM America/New_York
- Link/Location: https://example.com/kickoff-meeting
- Agenda: 
  - Introductions
  - Project goals, scope, and success criteria
  - Roles and responsibilities
  

In [23]:
from langgraph.types import Command

response = agent.invoke(
    Command(
        resume={"decisions": [{"type": "approve"}]}
    ),
    config=config # Same thread ID to resume the paused conversation
)

print(response["messages"][-1].content)

Email sent successfully to jude@example.com with the subject: Kickoff for Atlas Marketing Initiative – Welcome Aboard.

Summary of the email content:
- Project: Atlas Marketing Initiative
- Objective: Launch a coordinated marketing campaign for Q2
- Key milestones: Charter by Feb 20, 2026; Content assets by Feb 28, 2026; Go-live by Mar 15, 2026
- Your role: Marketing Analyst, reporting to Maria Chen
- Team: Maria Chen, Carlos Diaz, Priya Singh
- Kickoff meeting: Feb 12, 2026, 10:00 AM America/New_York, link provided
- Next steps: Review charter, RSVP by replying, share questions by Feb 9, 2026
- Info needed by kickoff: access to tools/repo, blockers, documents to share
- Sign-off: Please reply to confirm receipt or ask for tweaks
- Sender: Alex Carter, Project Manager

Would you like me to:
- Send a follow-up reminder to Jude if we don’t hear back by a certain date?
- Schedule a calendar invite for the kickoff meeting and send it to Jude?
- Check Jude’s inbox for a reply and draft a fo

In [25]:
from pprint import pprint

pprint(response)

{'authenticated': True,
 'messages': [HumanMessage(content='draft 1', additional_kwargs={}, response_metadata={}, id='22a7cafe-a99d-41f6-8604-c807b0fa7ead'),
              AIMessage(content='Could you clarify what you want drafted as “Draft 1”? Here are some quick options I can provide, plus what I’d need from you:\n\nPossible draft types\n- Email (welcome, follow-up, announcement)\n- Meeting agenda or minutes\n- Project proposal or plan\n- Policy or procedure document\n- Contract or NDA (short form)\n- Script or presentation notes\n\nInformation I need\n- Type of draft\n- Purpose and audience\n- Key points or sections to include\n- Desired length (word count or pages)\n- Tone (formal, neutral, friendly)\n- Any deadline or delivery date\n\nIf you’d like, I can also provide a ready-to-use generic “Draft 1” for a common item (e.g., a welcome email or a project kickoff email) and you can customize from there. Which would you prefer?', additional_kwargs={'refusal': None}, response_metadata

Langsmith Trace - https://smith.langchain.com/public/db96aadf-df03-4f85-b856-5bc59ce45bbc/r